# Predictive Maintenance — Reference Consumer

**Domain**: `adp-{stage}-foundation-domain` (us-east-1)
**Products consumed**: `vehicle_telemetry_aggregated` + `service_records` + `charging_sessions` + `energy_usage`
**Model**: `sklearn.ensemble.IsolationForest` (unsupervised anomaly detection)
**Output**: top-N at-risk-VIN report with per-VIN feature contributions

This notebook is the reference consumer for the EV-startup predictive-maintenance
use case. It runs end-to-end in **SageMaker Studio** attached to the ADP foundation
DataZone domain — no custom CDK, no Step Functions, no extra infra. The notebook
pulls 90 days of joined data from Athena, derives EV-relevant risk features (SoH
decline rate, charging-pattern anomalies, thermal stress), trains an
`IsolationForest`, and emits a ranked at-risk-VIN report you can hand to a service
operations team.

## What the notebook does

1. **Subscribe checklist** — confirms the four required DataZone subscriptions
   are granted to the Studio project (manual prerequisite documented below).
2. **Athena helper** — `awswrangler.athena.read_sql_query` returning a pandas
   `DataFrame`. Falls back to a small `boto3` polling helper when
   `awswrangler` is not present.
3. **Joined query** — single Athena Trino query (Engine V3) that joins all four
   products on `vin` and aggregates per-VIN over a rolling 90-day window.
   Iceberg hidden-partition pruning happens via predicates on the partition
   columns (`event_date`, `service_date`, `session_date`, `usage_date`).
4. **Feature engineering** — derives composite EV-risk signals:
   - `soh_decline_pp_per_day` (Trino `regr_slope` × −1, percentage points/day)
   - `pct_dcfc_sessions`, `pct_interrupted_sessions`, `avg_duration_s`
   - `peak_pack_temp_c`, `avg_pack_temp_drive_c` (thermal stress under load)
   - `avg_efficiency_kwh_per_100mi` (energy efficiency, post-OTA drift visible)
5. **Model fit + score** — `IsolationForest(contamination=0.05, random_state=42)`
   on a standardized feature matrix. Decision-function output flipped so higher
   = riskier.
6. **Top-N report** — top-`N` at-risk VINs ranked by anomaly score, with the
   three most-anomalous features called out per VIN (z-score against the fleet
   median).
7. **(Optional) Save** — writes the report to the project's default S3 prefix
   so downstream consumers (e.g., a Bedrock agent) can ingest it.

## Why these signals

| Signal | Why it matters for an EV startup |
|---|---|
| SoH decline rate | Battery degradation outpacing the fleet average is the single highest-leverage warranty signal. |
| Charging-pattern anomalies | Frequent DC-fast sessions or repeated interrupts correlate with thermal cycling and BMS faults. |
| Thermal stress under drive | High `battery_pack_temp_avg_c` while `is_charging=false` exposes liquid-cooling or BMS issues. |
| Service history | VINs already with `charging_system` / `hv_battery_diagnostic` / `battery_replacement` visits are a labelled positive prior. |
| Efficiency drift | `total_kwh_consumed / total_miles_driven` increasing vs the fleet median signals motor / inverter / accessory-load anomalies. |


## Prerequisites

### 1. SageMaker Studio with DataZone integration

You must run this notebook in a Studio space attached to a DataZone project that
is a member of the foundation domain `adp-{stage}-foundation-domain`. The
project's auto-provisioned IAM role grants the actions listed in
[`docs/cvx-integration-contract.md` §2.1](../../../docs/cvx-integration-contract.md#21-required-iam-actions-on-the-cvx-side-principal):

- `datazone:*` (search listings, create/get subscription requests, list grants)
- `glue:GetDatabase|GetDatabases|GetTable|GetTables|GetPartitions`
- `athena:StartQueryExecution|GetQueryExecution|GetQueryResults|StopQueryExecution`
- `lakeformation:GetDataAccess`
- `kms:Decrypt|DescribeKey` on `alias/adp-{stage}-foundation-lake`

### 2. Subscriptions

Subscribe the project to the four required listings **before** running the
notebook. The DataZone CLI pattern is identical for each product
(substitute `<PRODUCT>`):

```bash
export STAGE=staging        # or prod
export REGION=us-east-1
DOMAIN_ID=$(aws cloudformation list-exports --region "$REGION" \
  --query "Exports[?Name=='adp-${STAGE}-foundation-datazone-domain-id'].Value | [0]" \
  --output text)
PROJECT_ID=$(aws datazone list-projects --domain-identifier "$DOMAIN_ID" \
  --region "$REGION" --query 'items[?name==`my-pm-consumer-project`].id | [0]' \
  --output text)
for PRODUCT in vehicle_telemetry_aggregated service_records charging_sessions energy_usage; do
  ASSET_ID=$(aws datazone search-listings --domain-identifier "$DOMAIN_ID" \
    --search-text "$PRODUCT" --region "$REGION" \
    --query 'items[0].assetListing.entityId' --output text)
  REQ_ID=$(aws datazone create-subscription-request \
    --domain-identifier "$DOMAIN_ID" \
    --request-reason "Predictive maintenance reference consumer: $PRODUCT" \
    --subscribed-listings "{\"identifier\":\"$ASSET_ID\"}" \
    --subscribed-principals "{\"project\":{\"identifier\":\"$PROJECT_ID\"}}" \
    --region "$REGION" --query 'id' --output text)
  aws datazone accept-subscription-request \
    --domain-identifier "$DOMAIN_ID" --identifier "$REQ_ID" --region "$REGION"
done
```

Lake Formation grants typically propagate within ~30 seconds of acceptance.

### 3. Python packages

The SageMaker Distribution image ships `awswrangler`, `pandas`, `numpy`,
`scikit-learn`, and `boto3` by default. If you're on a slimmer kernel, run the
install cell below.


In [ ]:
# Optional: install missing packages on slimmer kernels.
# Comment this cell out on the SageMaker Distribution image — all
# required packages are pre-installed there.
# %pip install -q "awswrangler>=3.9" "pandas>=2.0" "scikit-learn>=1.3" "boto3>=1.34"

## 1. Configuration

All knobs are driven by environment variables so the notebook works unchanged
against `staging` and `prod` and against any custom analysis window or fleet
cohort. Defaults match the canonical staging deploy.


In [ ]:
import os
import datetime as dt
import json
import time
from typing import Optional

import boto3
import numpy as np
import pandas as pd

# ---- knobs ---------------------------------------------------------------
STAGE: str = os.environ.get("ADP_STAGE", "staging")              # staging | prod
REGION: str = os.environ.get("AWS_REGION", "us-east-1")
WINDOW_DAYS: int = int(os.environ.get("ADP_PM_WINDOW_DAYS", "90"))
END_DATE: dt.date = dt.date.fromisoformat(
    os.environ.get("ADP_PM_END_DATE", dt.date.today().isoformat())
)
START_DATE: dt.date = END_DATE - dt.timedelta(days=WINDOW_DAYS)
TOP_N: int = int(os.environ.get("ADP_PM_TOP_N", "50"))
CONTAMINATION: float = float(os.environ.get("ADP_PM_CONTAMINATION", "0.05"))
MIN_TELEM_SAMPLES: int = int(os.environ.get("ADP_PM_MIN_SAMPLES", "10"))

# Athena workgroup must be one the project's IAM role can use. SageMaker Studio
# projects with DataZone integration provision a per-project Athena workgroup.
# Fall back to "primary" if the project has not been wired with a custom one.
ATHENA_WORKGROUP: str = os.environ.get("ADP_ATHENA_WORKGROUP", "primary")

# Where Athena writes query results. Leave blank to use the workgroup's
# configured output location (recommended for project-scoped workgroups).
ATHENA_OUTPUT_S3: Optional[str] = os.environ.get("ADP_ATHENA_OUTPUT_S3") or None

# Optional: where to save the at-risk report. Leave blank to skip the save step.
REPORT_OUTPUT_S3: Optional[str] = os.environ.get("ADP_PM_REPORT_S3") or None

# ---- derived -------------------------------------------------------------
GLUE_TELEM   = f"adp_{STAGE}_vehicle_telemetry_aggregated.vehicle_telemetry_aggregated"
GLUE_SVC     = f"adp_{STAGE}_service_records.service_records"
GLUE_CHARGE  = f"adp_{STAGE}_charging_sessions.charging_sessions"
GLUE_ENERGY  = f"adp_{STAGE}_energy_usage.energy_usage"

print(f"Stage:            {STAGE}")
print(f"Region:           {REGION}")
print(f"Analysis window:  {START_DATE.isoformat()} → {END_DATE.isoformat()} ({WINDOW_DAYS} days)")
print(f"Athena workgroup: {ATHENA_WORKGROUP}")
print(f"Top-N report:     {TOP_N} VINs")
print(f"Contamination:    {CONTAMINATION}")

## 2. Athena helper

Use `awswrangler.athena.read_sql_query` when present (the canonical pattern in
SageMaker Studio Distribution). Fall back to a minimal `boto3` polling helper
that reads the result CSV directly so the notebook still works on slimmer
kernels.


In [ ]:
try:
    import awswrangler as wr  # noqa: F401
    _HAVE_WRANGLER = True
except Exception:
    _HAVE_WRANGLER = False

print(f"awswrangler available: {_HAVE_WRANGLER}")


def athena_query(sql: str) -> pd.DataFrame:
    """Run an Athena query and return a pandas DataFrame.

    The query runs against the workgroup configured in `ATHENA_WORKGROUP`.
    When `ATHENA_OUTPUT_S3` is unset, we rely on the workgroup's
    configured output location (the standard pattern for project-scoped
    workgroups in DataZone-integrated SageMaker Studio).
    """
    if _HAVE_WRANGLER:
        import awswrangler as wr  # local import; pre-checked above
        return wr.athena.read_sql_query(
            sql=sql,
            database=None,                 # fully-qualified Glue refs in SQL
            workgroup=ATHENA_WORKGROUP,
            s3_output=ATHENA_OUTPUT_S3,
            ctas_approach=False,           # streaming results, no CTAS
            boto3_session=boto3.Session(region_name=REGION),
        )

    # boto3 fallback ------------------------------------------------------
    if not ATHENA_OUTPUT_S3:
        raise RuntimeError(
            "boto3 fallback requires ADP_ATHENA_OUTPUT_S3 to be set "
            "(workgroup-default output not honored without awswrangler)."
        )
    athena = boto3.client("athena", region_name=REGION)
    s3 = boto3.client("s3", region_name=REGION)

    qid = athena.start_query_execution(
        QueryString=sql,
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT_S3},
        WorkGroup=ATHENA_WORKGROUP,
    )["QueryExecutionId"]

    while True:
        qe = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = qe["Status"]["State"]
        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = qe["Status"].get("StateChangeReason", "<no reason>")
            raise RuntimeError(f"Athena query {qid} {state}: {reason}")
        time.sleep(2)

    # Stream the result CSV from S3 directly into pandas.
    out_uri = qe["ResultConfiguration"]["OutputLocation"]
    assert out_uri.startswith("s3://"), out_uri
    bucket, key = out_uri[len("s3://"):].split("/", 1)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"]
    return pd.read_csv(body)

## 3. Joined Athena query — four products, 90-day window, per-VIN aggregates

The query uses Trino (Athena Engine V3) features:
- `regr_slope(y, x)` — linear-regression slope of SoH on day-offset; negate to
  surface decline as a positive number.
- `count_if(predicate)` — counts rows satisfying a boolean predicate without
  `CASE WHEN`.
- `filter` clause on aggregates — `AVG(...) FILTER (WHERE is_charging = false)`
  isolates drive-segment thermal samples from charging-segment samples.

Iceberg hidden-partition pruning kicks in on every product because each WHERE
clause filters on the partition column declared in the schema YAML
(`event_date` / `service_date` / `session_date` / `usage_date`).


In [ ]:
SQL_PER_VIN = f"""
WITH
telem_per_vin AS (
  SELECT
    v.vin,
    -- SoH decline rate as positive percentage points per day
    -- (regr_slope returns dy/dx; negate so a falling SoH -> positive value).
    -- date_diff('day', DATE '2000-01-01', event_date) gives a stable day index.
    -1.0 * regr_slope(
      v.state_of_health_pct,
      CAST(date_diff('day', DATE '2000-01-01', v.event_date) AS double)
    )                                                AS soh_decline_pp_per_day,
    AVG(v.state_of_health_pct)                       AS avg_soh_pct,
    MIN(v.state_of_health_pct)                       AS min_soh_pct,
    AVG(v.battery_pack_temp_avg_c)
        FILTER (WHERE v.is_charging = false)         AS avg_pack_temp_drive_c,
    MAX(v.battery_pack_temp_avg_c)                   AS peak_pack_temp_c,
    AVG(v.ambient_temp_avg_c)                        AS avg_ambient_c,
    AVG(v.motor_temp_c)                              AS avg_motor_temp_c,
    COUNT(*)                                         AS telem_samples
  FROM {GLUE_TELEM} v
  WHERE v.event_date BETWEEN DATE '{START_DATE.isoformat()}'
                        AND DATE '{END_DATE.isoformat()}'
  GROUP BY v.vin
),
charge_per_vin AS (
  SELECT
    cs.vin,
    COUNT(*)                                          AS sessions_total,
    count_if(cs.station_type = 'public_dc_fast')      AS sessions_dcfc,
    count_if(cs.interrupted)                          AS sessions_interrupted,
    AVG(cs.kwh_delivered)                             AS avg_kwh_per_session,
    AVG(CAST(cs.duration_seconds AS double))          AS avg_duration_s,
    AVG(cs.peak_power_kw)                             AS avg_peak_power_kw,
    AVG(cs.end_soc_pct - cs.start_soc_pct)            AS avg_soc_delta_pct
  FROM {GLUE_CHARGE} cs
  WHERE cs.session_date BETWEEN DATE '{START_DATE.isoformat()}'
                          AND DATE '{END_DATE.isoformat()}'
  GROUP BY cs.vin
),
energy_per_vin AS (
  SELECT
    eu.vin,
    AVG(eu.efficiency_kwh_per_100mi)   AS avg_efficiency_kwh_per_100mi,
    SUM(eu.total_kwh_consumed)         AS total_kwh_consumed,
    SUM(eu.total_kwh_charged)          AS total_kwh_charged,
    SUM(eu.total_miles_driven)         AS total_miles_driven,
    AVG(CAST(eu.battery_age_days AS double)) AS avg_battery_age_days,
    AVG(eu.regen_kwh_recovered)        AS avg_regen_kwh
  FROM {GLUE_ENERGY} eu
  WHERE eu.usage_date BETWEEN DATE '{START_DATE.isoformat()}'
                          AND DATE '{END_DATE.isoformat()}'
  GROUP BY eu.vin
),
service_per_vin AS (
  SELECT
    sr.vin,
    COUNT(*)                                                         AS service_visits,
    count_if(sr.service_type IN (
      'charging_system', 'battery_replacement', 'hv_battery_diagnostic'
    ))                                                               AS service_visits_battery_or_charging
  FROM {GLUE_SVC} sr
  WHERE sr.service_date BETWEEN DATE '{START_DATE.isoformat()}'
                          AND DATE '{END_DATE.isoformat()}'
  GROUP BY sr.vin
)
SELECT
  t.vin,
  t.soh_decline_pp_per_day,
  t.avg_soh_pct,
  t.min_soh_pct,
  t.avg_pack_temp_drive_c,
  t.peak_pack_temp_c,
  t.avg_ambient_c,
  t.avg_motor_temp_c,
  t.telem_samples,
  COALESCE(c.sessions_total, 0)                       AS sessions_total,
  COALESCE(c.sessions_dcfc, 0)                        AS sessions_dcfc,
  COALESCE(c.sessions_interrupted, 0)                 AS sessions_interrupted,
  COALESCE(c.avg_kwh_per_session, 0)                  AS avg_kwh_per_session,
  COALESCE(c.avg_duration_s, 0)                       AS avg_duration_s,
  COALESCE(c.avg_peak_power_kw, 0)                    AS avg_peak_power_kw,
  COALESCE(c.avg_soc_delta_pct, 0)                    AS avg_soc_delta_pct,
  COALESCE(e.avg_efficiency_kwh_per_100mi, 0)         AS avg_efficiency_kwh_per_100mi,
  COALESCE(e.total_kwh_consumed, 0)                   AS total_kwh_consumed,
  COALESCE(e.total_kwh_charged, 0)                    AS total_kwh_charged,
  COALESCE(e.total_miles_driven, 0)                   AS total_miles_driven,
  COALESCE(e.avg_battery_age_days, 0)                 AS avg_battery_age_days,
  COALESCE(e.avg_regen_kwh, 0)                        AS avg_regen_kwh,
  COALESCE(s.service_visits, 0)                       AS service_visits,
  COALESCE(s.service_visits_battery_or_charging, 0)   AS service_visits_battery_or_charging
FROM telem_per_vin t
LEFT JOIN charge_per_vin   c ON c.vin = t.vin
LEFT JOIN energy_per_vin   e ON e.vin = t.vin
LEFT JOIN service_per_vin  s ON s.vin = t.vin
WHERE t.telem_samples >= {MIN_TELEM_SAMPLES}
"""

print(f"Querying Athena ({STAGE}, window {START_DATE} → {END_DATE})…")
print(f"  Estimated scan ≤ ~3 GB at 100M-row telemetry × 90 days (Iceberg pruning).")

df_features = athena_query(SQL_PER_VIN)
print(f"\nVINs returned: {len(df_features):,}")
if len(df_features) == 0:
    print(
        "\n[!] Zero VINs returned. Likely causes:\n"
        "    - Subscriptions not granted yet (Lake Formation propagation ≤30s).\n"
        "    - The four products are empty in this stage (run `make seed STAGE=staging`).\n"
        "    - The analysis window has no data (try setting ADP_PM_END_DATE)."
    )
else:
    display(df_features.head())

## 4. Feature engineering

From the per-VIN aggregates we derive the modelling matrix. Three buckets:

- **SoH**: `soh_decline_pp_per_day`, `avg_soh_pct`. A high decline rate combined
  with low absolute SoH is a stronger signal than either alone.
- **Charging anomalies**: `pct_dcfc_sessions`, `pct_interrupted_sessions`,
  `avg_duration_s`. Frequent short-and-failed sessions are a different
  failure mode from frequent long-DCFC sessions.
- **Thermal stress**: `peak_pack_temp_c`, `avg_pack_temp_drive_c`. Elevated
  pack temperature *during drive* (not during charging) flags cooling issues.
- **Efficiency drift**: `avg_efficiency_kwh_per_100mi`. Above-fleet-median
  efficiency-loss after correcting for battery age suggests motor / inverter
  / accessory anomalies.
- **Service prior**: `service_visits_battery_or_charging`. Treated as a soft
  label — a VIN with prior battery-or-charging visits is a labelled-positive
  signal we keep in the feature matrix so IsolationForest treats it as a
  contributing dimension.


In [ ]:
if len(df_features) == 0:
    raise SystemExit("No VINs to score; resolve the prerequisite issues above and re-run.")

df = df_features.copy()

# Avoid divide-by-zero on VINs with no charging activity.
df["pct_dcfc_sessions"] = np.where(
    df["sessions_total"] > 0,
    df["sessions_dcfc"] / df["sessions_total"],
    0.0,
)
df["pct_interrupted_sessions"] = np.where(
    df["sessions_total"] > 0,
    df["sessions_interrupted"] / df["sessions_total"],
    0.0,
)

# Replace any null SoH-decline rate (single-sample VINs) with 0 so they
# neither help nor hurt the score.
df["soh_decline_pp_per_day"] = df["soh_decline_pp_per_day"].fillna(0.0)

FEATURE_COLS = [
    "soh_decline_pp_per_day",
    "avg_soh_pct",
    "min_soh_pct",
    "peak_pack_temp_c",
    "avg_pack_temp_drive_c",
    "pct_dcfc_sessions",
    "pct_interrupted_sessions",
    "avg_duration_s",
    "avg_efficiency_kwh_per_100mi",
    "avg_battery_age_days",
    "service_visits_battery_or_charging",
]
# Static lint: every FEATURE_COLS entry must exist in df.columns. If the
# SQL above changes (column rename, dropped CTE, etc.) and FEATURE_COLS
# drifts out of sync, fail fast with a clear message instead of letting
# the median-fill loop raise a cryptic KeyError mid-iteration.
missing = [c for c in FEATURE_COLS if c not in df.columns]
assert not missing, f"FEATURE_COLS not in SQL output: {missing}"

# Replace any remaining NaNs with column medians (post-merge holes from
# left joins on VINs with no charging or no energy rows).
for col in FEATURE_COLS:
    med = df[col].median()
    if pd.isna(med):
        med = 0.0
    df[col] = df[col].fillna(med)

print(f"Feature matrix: {len(df):,} VINs × {len(FEATURE_COLS)} features")
df[FEATURE_COLS].describe().T

## 5. Train IsolationForest

`IsolationForest` is the canonical sklearn unsupervised anomaly detector. It
builds an ensemble of random partitioning trees; the harder a point is to
isolate, the longer the average path length, the lower the anomaly score.

We use:
- `contamination=0.05` — assume ~5% of fleet VINs are at risk.
- `n_estimators=200` — stable at this fleet size; scales linearly.
- `random_state=42` — reproducible across runs (the entire ADP foundation
  pins seed=42 for synthetic-data reproducibility).
- Standardize features first so each dimension contributes proportionally.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

X = df[FEATURE_COLS].to_numpy(dtype=float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iforest = IsolationForest(
    n_estimators=200,
    contamination=CONTAMINATION,
    random_state=42,
    n_jobs=-1,
)
iforest.fit(X_scaled)

# decision_function: higher = more normal. Negate so higher = riskier.
df["risk_score"] = -iforest.decision_function(X_scaled)
df["is_anomaly"] = (iforest.predict(X_scaled) == -1).astype(int)

print(f"Anomalies flagged: {int(df['is_anomaly'].sum())} / {len(df)} "
      f"({df['is_anomaly'].mean():.1%})")
print(f"Risk score quantiles: "
      f"min={df['risk_score'].min():.3f}, "
      f"p50={df['risk_score'].median():.3f}, "
      f"p95={df['risk_score'].quantile(0.95):.3f}, "
      f"max={df['risk_score'].max():.3f}")

## 6. Top-N at-risk-VIN report

Surface the top `TOP_N` VINs ranked by `risk_score`. For each VIN we also
list the three features whose z-score against the fleet median is highest
in absolute value — that's a cheap, model-agnostic interpretability signal
that lets the service team know *why* a VIN ranked.


In [ ]:
# Per-feature z-scores (against fleet median + MAD-ish std).
feature_medians = df[FEATURE_COLS].median()
feature_stds = df[FEATURE_COLS].std(ddof=0).replace(0.0, 1.0)
z = (df[FEATURE_COLS] - feature_medians) / feature_stds

def top_drivers(row: pd.Series, k: int = 3) -> str:
    """Return a comma-separated list of the k features with the most
    extreme z-score on this row."""
    ranked = row.abs().sort_values(ascending=False).head(k)
    parts = []
    for col in ranked.index:
        sign = "+" if row[col] >= 0 else "-"
        parts.append(f"{col}({sign}{abs(row[col]):.1f}σ)")
    return ", ".join(parts)

df["top_drivers"] = z.apply(top_drivers, axis=1)

REPORT_COLS = [
    "vin",
    "risk_score",
    "is_anomaly",
    "soh_decline_pp_per_day",
    "avg_soh_pct",
    "peak_pack_temp_c",
    "pct_dcfc_sessions",
    "pct_interrupted_sessions",
    "avg_efficiency_kwh_per_100mi",
    "service_visits_battery_or_charging",
    "top_drivers",
]
report = (
    df.sort_values("risk_score", ascending=False)
      .head(TOP_N)
      .loc[:, REPORT_COLS]
      .reset_index(drop=True)
)

print(f"Top {TOP_N} at-risk VINs ({STAGE}, {START_DATE} → {END_DATE}):")
display(report)

## 7. (Optional) Save the report to S3

When `ADP_PM_REPORT_S3` is set (e.g.,
`s3://adp-{stage}-foundation-lake-<account>-us-east-1/reference-consumers/predictive-maintenance/`),
we persist the report as both CSV and parquet so a downstream Bedrock agent or
QuickSight dashboard can ingest it. The IAM grant for this write must be
on the project's S3 prefix — DataZone-managed grants do **not** include write
on the lake bucket itself; the project's default S3 prefix is the right
target. Skip this cell if no output prefix is configured.


In [ ]:
if not REPORT_OUTPUT_S3:
    print("ADP_PM_REPORT_S3 not set — skipping save step. "
          "Set the env var to an S3 prefix you have write access to.")
else:
    prefix = REPORT_OUTPUT_S3.rstrip("/")
    run_id = dt.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    csv_uri = f"{prefix}/run_id={run_id}/at_risk_vins.csv"
    pq_uri  = f"{prefix}/run_id={run_id}/at_risk_vins.parquet"

    # awswrangler if available; otherwise to_csv + boto3 put_object.
    if _HAVE_WRANGLER:
        import awswrangler as wr
        wr.s3.to_csv(report, path=csv_uri, index=False,
                     boto3_session=boto3.Session(region_name=REGION))
        wr.s3.to_parquet(report, path=pq_uri,
                         boto3_session=boto3.Session(region_name=REGION))
    else:
        import io as _io
        s3 = boto3.client("s3", region_name=REGION)
        assert csv_uri.startswith("s3://"), csv_uri
        bkt, key = csv_uri[len("s3://"):].split("/", 1)
        buf = _io.StringIO()
        report.to_csv(buf, index=False)
        s3.put_object(Bucket=bkt, Key=key, Body=buf.getvalue().encode("utf-8"))
        print(f"  CSV  → {csv_uri}")
        print(f"  (parquet save skipped without awswrangler)")
        pq_uri = None

    print("Saved at-risk report:")
    print(f"  CSV     → {csv_uri}")
    if pq_uri:
        print(f"  parquet → {pq_uri}")

    # Manifest a tiny JSON descriptor so consumers can find the latest run.
    manifest = {
        "run_id": run_id,
        "stage": STAGE,
        "region": REGION,
        "window_start": START_DATE.isoformat(),
        "window_end": END_DATE.isoformat(),
        "top_n": TOP_N,
        "contamination": CONTAMINATION,
        "vin_count_total": int(len(df)),
        "vin_count_anomaly": int(df["is_anomaly"].sum()),
        "feature_cols": FEATURE_COLS,
        "csv_uri": csv_uri,
        "parquet_uri": pq_uri,
    }
    manifest_uri = f"{prefix}/run_id={run_id}/manifest.json"
    s3 = boto3.client("s3", region_name=REGION)
    bkt, key = manifest_uri[len("s3://"):].split("/", 1)
    s3.put_object(
        Bucket=bkt, Key=key,
        Body=json.dumps(manifest, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    print(f"  manifest → {manifest_uri}")

## What's next

- **Hand the top-N report to the service operations team** as a triage queue.
  The `top_drivers` column tells the operator *which* dimension(s) drove the
  ranking — pack-temp anomalies route to a thermal/cooling diagnostic, charging
  anomalies to a BMS/charger diagnostic, SoH decline to a warranty review.
- **Iterate the feature set**. The `vehicle_identity` and `ota_campaigns`
  products carry strong covariates — battery chemistry, max charging rate,
  recent OTA install timing — that we deliberately omitted from v1 to keep
  the IAM/subscription scope minimal. Add them once the consumer project's
  subscriptions widen.
- **Productionize**. Treat this notebook as the offline reference; promote
  the SQL + featurization to a scheduled job (Airflow / EventBridge +
  SageMaker Processing) once the model definition stabilises. Custom CDK
  and Step Functions infrastructure are intentionally **out of scope** for
  this reference consumer — the foundation already provides the data plane,
  and a scheduled job is a single ProcessingStep, not a bespoke stack.
